# اجرای ویترین‌یاب در Google Colab
این نوت‌بوک Laravel و FastAPI را بدون Docker اجرا می‌کند و در پایان یک لینک HTTPS موقت نمایش می‌دهد. چون مخزن Public است، به GitHub Token نیاز ندارید.

In [ ]:
# Clone the public repository
import os, shutil, subprocess
REPOSITORY = 'https://github.com/Alirezaab78/clothes_search.git'
PROJECT_DIR = '/content/clothes_search'
if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)
subprocess.run(['git', 'clone', '--depth', '1', REPOSITORY, PROJECT_DIR], check=True)
print('✅ Project cloned:', PROJECT_DIR)

✅ Project cloned: /content/clothes_search


In [ ]:

%%bash
PROJECT_DIR=/content/clothes_search
export DEBIAN_FRONTEND=noninteractive

# ۱. نصب پیش‌نیازهای لینوکسی و PHP
apt-get update -qq
apt-get install -y -qq php-cli php-curl php-mbstring php-xml php-zip php-sqlite3 unzip curl git

if ! command -v composer >/dev/null; then
  curl -fsSL https://getcomposer.org/installer -o /tmp/composer-setup.php
  php /tmp/composer-setup.php --install-dir=/usr/local/bin --filename=composer --quiet
fi

# ۲. نصب نیازمندی‌های هوش مصنوعی و اجرای FastAPI
cd $PROJECT_DIR/ai-service
pip install -r requirements.txt --quiet --no-warn-conflicts
nohup uvicorn app.main:app --host 127.0.0.1 --port 8001 >/tmp/fashion-ai.log 2>&1 &

# ۳. راه‌اندازی لاراول با دور زدن محدودیت نسخه PHP
cd $PROJECT_DIR/laravel-app
composer install --no-interaction --prefer-dist --optimize-autoloader --ignore-platform-reqs -q

cp -n .env.example .env || true
touch database/database.sqlite
sed -i 's|^DB_CONNECTION=.*|DB_CONNECTION=sqlite|' .env
sed -i 's|^FASHION_AI_URL=.*|FASHION_AI_URL=http://127.0.0.1:8001|' .env
grep -q '^FASHION_AI_URL=' .env || echo 'FASHION_AI_URL=http://127.0.0.1:8001' >> .env

php artisan key:generate --force
php artisan migrate --force
php artisan storage:link || true
nohup php artisan serve --host=127.0.0.1 --port=8000 >/tmp/laravel.log 2>&1 &

# ۴. بررسی بالا آمدن سرورها
echo "Waiting for services to spin up..."
for i in $(seq 1 120); do curl -fs http://127.0.0.1:8001/health >/dev/null 2>&1 && break; sleep 1; done
for i in $(seq 1 30); do curl -fs http://127.0.0.1:8000 >/dev/null 2>&1 && break; sleep 1; done

echo "✅ All services (FastAPI & Laravel) are running successfully!"




   INFO  Application key set successfully.  


   INFO  Preparing database.  

  Creating migration table ...................................... 18.38ms DONE

   INFO  Running migrations.  

  0001_01_01_000000_create_users_table .......................... 37.97ms DONE
  0001_01_01_000001_create_cache_table .......................... 11.80ms DONE
  0001_01_01_000002_create_jobs_table ........................... 29.38ms DONE
  2026_09_14_000100_add_phone_and_search_credit_to_users_table .. 15.52ms DONE
  2026_09_14_000200_create_shops_table ........................... 6.24ms DONE
  2026_09_14_000300_create_products_table ....................... 11.72ms DONE
  2026_09_14_000400_create_subscriptions_table .................. 12.96ms DONE
  2026_09_14_000500_create_search_transactions_table ............. 6.81ms DONE


   INFO  The [public/storage] link has been connected to [storage/app/public].  

Waiting for services to spin up...
✅ All services (FastAPI & Laravel) are running successful

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
cp: warning: behavior of -n is non-portable and may change in future; use --update=none instead


In [ ]:



%%bash
# دانلود و نصب کلاینت Cloudflare Tunnel
if [ ! -f /usr/local/bin/cloudflared ]; then
  curl -fsSL -L --retry 3 https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
  chmod +x /usr/local/bin/cloudflared
fi

# اجرای تونل به سمت پورت لاراول (8000)
pkill cloudflared || true
nohup cloudflared tunnel --url http://127.0.0.1:8000 >/tmp/cloudflared.log 2>&1 &

# استخراج لینک عمومی با تایم‌اوت مناسب
for i in $(seq 1 40); do
  URL=$(grep -oE 'https://[a-zA-Z0-9-]+\.trycloudflare\.com' /tmp/cloudflared.log | head -n 1 || true)
  if [ -n "$URL" ]; then
    break
  fi
  sleep 2
done

if [ -z "${URL:-}" ]; then
  echo "Tunnel URL was not created. Log:"
  cat /tmp/cloudflared.log
  exit 1
fi

echo ""
echo "🎉 سایت شما با موفقیت آنلاین شد!"
echo "🔗 لینک دسترسی: $URL"



🎉 سایت شما با موفقیت آنلاین شد!
🔗 لینک دسترسی: https://numbers-theology-intended-animated.trycloudflare.com


## استفاده
در یک Runtime تازه، سه سلول کد را به ترتیب یا با Runtime → Run all اجرا کنید. لینک `trycloudflare.com` در خروجی سلول آخر نشان داده می‌شود. این لینک با پایان Runtime نامعتبر می‌شود.

In [4]:
%%bash
PROJECT_DIR=/content/clothes_search/laravel-app

# ۱. ساخت دایرکتوری‌های کش و فریم‌ورک لاراول و دادن دسترسی کامل
mkdir -p $PROJECT_DIR/storage/framework/{sessions,views,cache,testing}
mkdir -p $PROJECT_DIR/storage/logs
mkdir -p $PROJECT_DIR/bootstrap/cache
chmod -R 777 $PROJECT_DIR/storage $PROJECT_DIR/bootstrap/cache

# ۲. پاک‌سازی کش‌های پیکربندی
cd $PROJECT_DIR
php artisan config:clear
php artisan view:clear
php artisan cache:clear

# ۳. ری‌استارت کردن وب‌سرور لاراول
pkill -f "artisan serve" || true
nohup php artisan serve --host=127.0.0.1 --port=8000 >/tmp/laravel.log 2>&1 &

sleep 2
echo "✅ کش‌ها ساخته شدند و سرور لاراول مجدداً ری‌استارت شد!"



   INFO  Configuration cache cleared successfully.  


   INFO  Compiled views cleared successfully.  


   INFO  Application cache cleared successfully.  

✅ کش‌ها ساخته شدند و سرور لاراول مجدداً ری‌استارت شد!
